# dmpbridge — Experiment Log

Records every experiment, its result, and the decision made.  
Use this as the reference before changing any pipeline component.

## Registry

| ID | Name | Status | Date | Key result |
|---|---|---|---|---|
| E01 | Input encoding: flags vs markdown | ✅ Adopted (flags) | 2026-05 | flags 88.3% vs markdown 80.0% |
| E02 | Few-shot examples | ✅ Adopted | 2026-05 | sec.description F1: 0% → 94.2% |
| E03 | Mid-line font splitting | ❌ Rejected (reverted) | 2026-05 | accuracy 97.2% → 92.5% |
| E04 | Bold/italic: first-char vs majority vote | ✅ Adopted (first-char) | 2026-05 | majority vote breaks Rule 1 |
| E05 | Smoothing Rule 1 (bold+italic → question.text) | ✅ Adopted | 2026-05 | defensive — 0 fires on 10 samples |
| E06 | Smoothing Rule 2 (italic continuation) | ✅ Adopted | 2026-05 | question.text F1: 77% → 90.5% |
| E07 | Model comparison: llama3.3:70b vs llama3.1:8b | 📌 Concluded | 2026-06 | 70b: 96.8%, 8b: 69.0% |
| E08 | Ablation: rules removed from both models | 📌 Concluded | 2026-06-24 | 70b drops 96.8→95.6%; 8b barely moves |
| E09 | Fine-tune llama3.1:8b on labeled data | 🔲 Planned | — | — |
| E10 | Sequence correction pass (state machine) | 🔲 Planned | — | — |
| E11 | Expand dataset: ERC/Wellcome/UKRI DMPs | 🔲 Planned | — | — |

**Status legend:** ✅ Adopted · ❌ Rejected · 📌 Concluded (informational) · 🔲 Planned

---
## E01 — Input encoding: Boolean flags vs Markdown

**Hypothesis:** Boolean flags (`"bold": true`) give the model cleaner signal than markdown encoding (`## **bold**`).  
**Setup:** Same 3 samples, same model (llama3.3:70b), same few-shot prompt. Only the block serialisation differed.  
**Result:**

| Encoding | Overall accuracy |
|---|---|
| Boolean flags (A1) | 88.3% |
| Markdown (A2) | 80.0% |

**Decision ✅ Adopted:** Boolean flags. Markdown encoding caused the model to confuse formatting markers with structural semantics.  
**File:** `dmpbridge/classifier.py` — block serialisation in `_format_block()`

---
## E02 — Few-shot examples

**Hypothesis:** Providing labeled DMP examples in the system prompt guides the model toward correct label usage, especially for funder-written text.  
**Setup:** llama3.3:70b on 3 samples, with and without few-shot examples in system prompt.  
**Result:**

| Setup | sec.description F1 | question.text F1 | Overall |
|---|---|---|---|
| No few-shot | 0% | ~50% | ~70% |
| With few-shot | 94.2% | 90.5% | 97.2% |

**Decision ✅ Adopted:** Few-shot examples are non-negotiable. Without them, `section.description` recall collapses entirely — the model defaults everything funder-written to `answer.text`.  
**File:** `dmpbridge/classifier.py` — system prompt construction

---
## E03 — Mid-line font splitting

**Hypothesis:** Splitting lines at mid-line font changes (e.g. bold label followed by normal answer on the same line) would improve label boundary precision.  
**Setup:** Modified `extractor.py` to split pdfplumber characters at bold/non-bold transitions within a line.  
**Result:**

| State | Total blocks | Overall accuracy | question.text recall |
|---|---|---|---|
| Before split | 726 | 97.2% | 92.7% |
| After split | 762 | 92.5% | 34.0% |
| After revert | 726 | 97.2% | 92.7% |

**Root cause:** pdfplumber `chars` has no space characters; splitting at font boundaries created fragments that lost syntactic context and were systematically mislabeled.  
**Decision ❌ Rejected and reverted.** One block per pdfplumber line is the correct granularity.  
**File:** `dmpbridge/extractor.py` — `extract_blocks()`

---
## E04 — Bold/italic detection: first-char vs majority vote

**Hypothesis:** Determining `is_bold` from the first non-whitespace character's font is more reliable than majority-vote across all characters in the line.  
**Setup:** Compared both methods on 10 samples. Specifically looked at lines like *"Content and Format. A statement of…"* where the label is bold but the answer is not.  
**Result:** Majority vote incorrectly marks mixed-style lines as non-bold, causing Rule 1 to miss them. First-char correctly identifies the heading prefix.  
**Decision ✅ Adopted:** First non-whitespace character's font.  
**File:** `dmpbridge/extractor.py` — `_char_style()` helper

---
## E05 — Smoothing Rule 1: bold+italic block → `question.text`

**Hypothesis:** Blocks that are both bold and italic but not numbered are researcher sub-questions, not funder section headings or descriptions.  
**Setup:** Added post-classification rule: if `is_bold AND is_italic AND label in (section.title, section.description) AND not numbered` → reclassify as `question.text`.  
**Result (on 10 samples):** Rule fires 0 times. No accuracy change observed. Rule is defensive — it guards against a pattern present in some DMP templates but absent from the current 10 samples.  
**Decision ✅ Adopted (defensive):** Rule costs nothing and prevents a known failure mode on DMP templates with bold+italic sub-questions (seen in ERC-style DMPs).  
**Ablation (E08):** Removing Rule 1 had 0% impact on accuracy for both models across all 10 samples.  
**File:** `dmpbridge/pipeline.py` — `_smooth_labels()` Rule 1

---
## E06 — Smoothing Rule 2: italic continuation → `question.text`

**Hypothesis:** An italic-only block following a `question.text` that is mislabeled `section.description` is a continuation of the question, not funder text.  
**Setup:** Added post-classification rule: if `is_italic AND NOT is_bold AND label==section.description AND prev==question.text` → reclassify as `question.text`.  
**Result:**

| Metric | Before Rule 2 | After Rule 2 |
|---|---|---|
| question.text recall | 69% | 92.7% |
| question.text F1 | ~77% | 90.5% |
| Blocks changed (10 samples) | — | 27 |

**Decision ✅ Adopted:** Rule 2 is the single largest accuracy contributor outside of few-shot examples. All 27 changes on the 10-sample set are correct.  
**Ablation (E08):** Without Rule 2, question.text F1 drops from 90.5% → 77.3% for llama3.3:70b. Rule has minimal effect on llama3.1:8b (already failing semantically).  
**File:** `dmpbridge/pipeline.py` — `_smooth_labels()` Rule 2

---
## E07 — Model comparison: llama3.3:70b vs llama3.1:8b

**Hypothesis:** llama3.1:8b can achieve acceptable accuracy on the DMP classification task, enabling deployment without 42 GB of RAM.  
**Setup:** Both models run on all 10 samples with smoothing enabled. Evaluated with `evaluate.py`.  
**Result:**

| Model | Overall accuracy | question.text F1 | sec.description F1 |
|---|---|---|---|
| llama3.3:70b | 96.8% (706/729) | 90.5% | 94.2% |
| llama3.1:8b | 69.0% (503/729) | 33.8% | 40.0% |
| Diff | −27.8% | −56.7% | −54.2% |

**Root cause:** llama3.1:8b cannot reliably distinguish funder-written instructions (`section.description`) from researcher sub-questions (`question.text`). Few-shot examples do not bridge this gap for the 8b model.  
**Decision 📌 Concluded:** llama3.1:8b is inadequate for production. llama3.3:70b is required. Fine-tuning llama3.1:8b is listed as E09 (planned).  
**Notebooks:** `eval_llama3.3-70b.ipynb`, `eval_llama3.1-8b.ipynb`, Section 7 of `03_label_evaluation.ipynb`

---
## E08 — Ablation: smoothing rules removed from both models

**Hypothesis:** The two smoothing rules add measurable accuracy on top of raw LLM output; removing them degrades results.  
**Setup:** Both models re-run with `--no-smooth` flag on all 10 samples (saved as `*-nosmooth.json`). Evaluated against same ground truth.  
**Result:**

| Variant | Accuracy | question.text F1 | sec.description F1 |
|---|---|---|---|
| llama3.3:70b + rules | 96.8% | 90.5% | 94.2% |
| llama3.3:70b no rules | 95.6% | 77.3% | 87.7% |
| llama3.1:8b + rules | 69.0% | 33.8% | 40.0% |
| llama3.1:8b no rules | 68.4% | 27.4% | 40.5% |

**Key findings:**
- Rule 1 fires 0 times on these 10 samples (defensive only)
- Rule 2 changes 27 blocks for 70b, accounting for the full 1.2% accuracy gain
- All 9 extra errors without rules for 70b are in sample2 (italic continuation pattern)
- Rules have negligible effect on 8b — its failure is semantic, not font-style

**Decision 📌 Concluded:** Rules are retained. They are cheap, correct, and give +13.1% F1 on question.text for the primary model.  
**Notebooks:** Section 8 of `03_label_evaluation.ipynb`, `eval_llama3.3-70b.ipynb` Section 6, `eval_llama3.1-8b.ipynb` Section 6

---
## E09 — Fine-tune llama3.1:8b  *(Planned)*

**Hypothesis:** Fine-tuning llama3.1:8b on the 10 labeled DMP samples (with augmentation) will close most of the 27.8% gap to llama3.3:70b, enabling deployment on commodity hardware.  
**Setup (planned):** Generate training pairs from `data/manuallabeled/`. Fine-tune with LoRA or QLoRA. Evaluate on held-out samples.  
**Expected outcome:** question.text F1 from 33.8% → 70%+ (hypothesis).  
**Blocker:** Need at least 3 held-out samples not used in fine-tuning — current dataset is only 10.  
**Decision 🔲 Planned:** Depends on E11 (expand dataset) first.

---
## E10 — Sequence correction pass  *(Planned)*

**Hypothesis:** A lightweight state machine over the predicted label sequence can fix transition violations (e.g. `answer.text → section.description` with no intervening `section.title`) and eliminate 8–10 of the remaining 23 errors.  
**Setup (planned):** Post-process predicted sequence with legal transition rules. No model inference required.  
**Expected outcome:** accuracy from 96.8% → 98%+ for llama3.3:70b.  
**Decision 🔲 Planned:** Low effort, high expected gain. Next pipeline improvement after E11.

---
## E11 — Expand dataset: ERC / Wellcome / UKRI DMPs  *(Planned)*

**Hypothesis:** Adding 10 DMPs from non-NIH/NSF funders will reveal whether the current rules and few-shot examples generalise or overfit to NIH/NSF formatting.  
**Setup (planned):** Manually label 10 additional PDFs from ERC, Wellcome Trust, UKRI. Re-evaluate pipeline.  
**Expected outcome:** Either confirms generalisation, or reveals new failure modes that drive further rule development.  
**Decision 🔲 Planned:** Prerequisite for E09 (need held-out samples).